# Qwen2.5-7B Continued Pretraining with Unsloth (Google Colab)

This notebook is optimized for **Google Colab** (free Tesla T4 GPU).

**Features:**
- Colab-optimized installation & environment
- Google Drive mounting for persistent checkpoint storage
- Automatic checkpointing during training (saves every N steps)
- Resume from checkpoint if Colab disconnects
- Model saving to Google Drive after training

<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>
</div>

---
### 0. Mount Google Drive
Checkpoints and the final model will be saved to Google Drive so they persist across Colab sessions.

In [ ]:
import os
try:
    import google.colab
    from google.colab import userdata, drive
    drive.mount('/content/drive')
    IS_COLAB = True
    token = userdata.get("HF_TOKEN")
    if not token:
      print(f"No Huggingface Token detected!")
except ImportError:
    IS_COLAB = False
    token = os.getenv("HF_TOKEN")


if IS_COLAB:
    OUTPUT_MODEL_NAME = f"Qwen2.5-0.5B-Instruct-CYSECFINETUNED-dataclean"  # the output model
    DRIVE_BASE        = "/content/drive/MyDrive/Colab Notebooks/ContinuedPretrainingLLM"  # root folder on Drive
    CHECKPOINT_DIR    = f"{DRIVE_BASE}/checkpoints/{OUTPUT_MODEL_NAME}"             # training checkpoints
    FINAL_MODEL_DIR   = f"{DRIVE_BASE}/final_model/{OUTPUT_MODEL_NAME}"             # merged model output
    LORA_MODEL_DIR    = f"{DRIVE_BASE}/lora_model/{OUTPUT_MODEL_NAME}"              # LoRA adapter output

    import os
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
    os.makedirs(LORA_MODEL_DIR, exist_ok=True)
    print(f"Checkpoint dir : {CHECKPOINT_DIR}")
    print(f"Final model dir: {FINAL_MODEL_DIR}")
    print(f"LoRA model dir : {LORA_MODEL_DIR}")
else:
    OUTPUT_MODEL_NAME = f"Qwen2.5-0.5B-Instruct-CYSECFINETUNED-dataclean"
    ROOT = None
    _ROOT_CANDIDATE = [
        "/mnt/d/Projects/CysecLLM/ContinuedPretrainingLLM",
        "/mnt/c/Users/ADITYA RM/Documents/code/TA/LLM",
        os.getcwd(),
    ]
    for roots in _ROOT_CANDIDATE:
        if os.path.exists(roots):
            ROOT = roots
            break

    CHECKPOINT_DIR    = f"{ROOT}/models/checkpoints/{OUTPUT_MODEL_NAME}"   
    FINAL_MODEL_DIR   = f"{ROOT}/models/finetuned/{OUTPUT_MODEL_NAME}_COMPLETE"
    LORA_MODEL_DIR    = f"{ROOT}/models/finetuned/{OUTPUT_MODEL_NAME}"

    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
    os.makedirs(LORA_MODEL_DIR, exist_ok=True)
    print(f"Checkpoint dir : {CHECKPOINT_DIR}")
    print(f"Final model dir: {FINAL_MODEL_DIR}")
    print(f"LoRA model dir : {LORA_MODEL_DIR}")

Checkpoint dir : /content/drive/MyDrive/Colab Notebooks/ContinuedPretrainingLLM/checkpoints/Qwen2.5-0.5B-Instruct-CYSECFINETUNED
Final model dir: /content/drive/MyDrive/Colab Notebooks/ContinuedPretrainingLLM/final_model/Qwen2.5-0.5B-Instruct-CYSECFINETUNED
LoRA model dir : /content/drive/MyDrive/Colab Notebooks/ContinuedPretrainingLLM/lora_model/Qwen2.5-0.5B-Instruct-CYSECFINETUNED


---
### 1. Installation
Install Unsloth and pinned dependency versions for Colab.

In [ ]:
%%capture
import os, re

# Colab-specific fast install
import torch
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

---
### 2. Load Model
Load **Qwen2.5-7B-Instruct** in 4-bit quantization to fit within the T4's 15 GB VRAM.

In [3]:
%env UNSLOTH_RETURN_LOGITS=1

env: UNSLOTH_RETURN_LOGITS=1


In [ ]:
from unsloth import FastLanguageModel
import torch, os

max_seq_length = 2048   # RoPE Scaling is handled automatically
dtype = None             # Auto-detect: Float16 for T4, Bfloat16 for Ampere+
load_in_4bit = True      # 4-bit quantization to reduce VRAM

# HF token — set via Colab Secrets or environment variable

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = token,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


---
### 3. Add LoRA Adapters
We include `embed_tokens` and `lm_head` for continual pretraining on out-of-distribution data.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "embed_tokens", "lm_head",        # for continual pretraining
    ],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # 30% less VRAM
    random_state = 3407,
    use_rslora = True,
    loftq_config = None,
)

Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


---
<a name="Data"></a>
### 4. Data Prep
Load the **Primus-Seed** cybersecurity dataset and append the EOS token.

In [ ]:
from datasets import load_dataset
from huggingface_hub import login
import os

# Log in to Hugging Face — required for gated datasets
if not token:
    login()   # will prompt interactively in Colab
else:
    login(token=token)

dataset = load_dataset(
    "trendmicro-ailab/Primus-Seed",
    "default",
    split="train",
)

print(f"number of cpu : {os.cpu_count()}")

def is_english(text):
    """Returns True if text is predominantly English (ASCII) characters."""
    if not text or not isinstance(text, str):
        return False
    
    total = len(text)
    if total == 0:
        return False
    
    ascii_chars = sum(1 for c in text if ord(c) < 128)
    ratio = ascii_chars / total
    
    return ratio >= 0.95  # 95% ASCII threshold

dataset = dataset.filter(
    lambda x : (
        is_english(x['content'])
        and len(x['content']) <= 478000
    ),
    num_proc=os.cpu_count(),
)

dataset = dataset.shuffle(seed=3407)

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    return {"content": [example + EOS_TOKEN for example in examples["content"]]}

dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"Dataset ready: {len(dataset)} rows")

In [16]:
# Preview first 5 samples
for row in dataset[:5]["content"]:
    print("=========================")
    print(row[:500])  # truncate for readability

# Understanding Cluster Deletion Protection for Amazon Aurora

As cloud engineers, our primary goal often revolves around ensuring the integrity and availability of our data. One critical aspect of this is implementing safeguards against accidental deletions, especially for vital resources like Amazon Aurora database clusters. This article delves into the concept of **Cluster Deletion Protection**, provides practical guidance on its implementation, and discusses best practices and potential pitf
# Securing Amazon SageMaker Notebook Instances in a VPC: A Guide for Security Engineers

## Introduction

As organizations increasingly adopt cloud computing, the need for robust security practices in machine learning operations becomes paramount. Amazon SageMaker, a fully managed service that provides every developer and data scientist with the ability to build, train, and deploy machine learning models quickly, offers powerful capabilities. However, leveraging these capabilities securely is c

---
<a name="Train"></a>
### 5. Training with Checkpointing

**Checkpointing strategy:**
- Saves a checkpoint every **100 steps** to Google Drive
- Keeps the **last 3 checkpoints** to conserve Drive space
- Automatically **resumes from the latest checkpoint** if one exists (e.g., after a Colab disconnect)

Set `embedding_learning_rate` to 10x smaller than `learning_rate` for stable continual pretraining.

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import UnslothTrainer, UnslothTrainingArguments

def formatting_func(example):
    """Handle both single-example (validation) and batched (training) calls."""
    content = example["content"]
    if isinstance(content, list):
        return content          # batched mode
    return [content]            # single example mode

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    formatting_func = formatting_func,
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,           # Colab typically has 2 CPU cores

    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,

        warmup_ratio = 0.1,
        num_train_epochs = 1,

        learning_rate = 5e-5,
        embedding_learning_rate = 5e-6,

        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),

        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.00,
        lr_scheduler_type = "cosine",
        seed = 3407,

        # --- Checkpoint settings ---
        output_dir = CHECKPOINT_DIR,          # save checkpoints to Google Drive
        save_strategy = "steps",              # save by step count
        save_steps = 500,                     # save every 100 steps
        save_total_limit = 3,                 # keep only the last 3 checkpoints
        save_only_model = False,              # save optimizer & scheduler state too (for resume)

        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/86987 [00:00<?, ? examples/s]

In [9]:
# Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.494 GB.
7.977 GB of memory reserved.


In [10]:
# --- Resume from checkpoint if one exists ---
import glob

checkpoint_dirs = sorted(
    glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*")),
    key=os.path.getmtime,
)

resume_from = checkpoint_dirs[-1] if checkpoint_dirs else None

if resume_from:
    print(f"Resuming training from checkpoint: {resume_from}")
else:
    print("No checkpoint found — starting training from scratch.")

trainer_stats = trainer.train(resume_from_checkpoint=resume_from)

No checkpoint found — starting training from scratch.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 86,987 | Num Epochs = 1 | Total steps = 5,437
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 1,412,956,160 of 9,028,572,672 (15.65% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.454500
2,1.678000
3,1.547600
4,1.676600
5,1.730600
6,1.574900
7,1.938100
8,1.690100
9,2.126000
10,1.457800


Step,Training Loss
1,1.454500
2,1.678000
3,1.547600
4,1.676600
5,1.730600
6,1.574900
7,1.938100
8,1.690100
9,2.126000
10,1.457800


In [11]:
# Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

26143.2283 seconds used for training.
435.72 minutes used for training.
Peak reserved memory = 25.082 GB.
Peak reserved memory for training = 17.105 GB.
Peak reserved memory % of max memory = 63.508 %.
Peak reserved memory for training % of max memory = 43.31 %.


---
### 6. Save Model to Google Drive
Save both the **LoRA adapter** (lightweight) and optionally the **merged full model**.

In [ ]:
# Save LoRA adapter (small, fast to save)
model.save_pretrained(LORA_MODEL_DIR)
tokenizer.save_pretrained(LORA_MODEL_DIR)
print(f"LoRA adapter saved to: {LORA_MODEL_DIR}")

In [ ]:
# Save merged 16-bit model (larger, but standalone — no base model needed)
# This can take a few minutes and requires extra VRAM
model.save_pretrained_merged(
    FINAL_MODEL_DIR,
    tokenizer,
    save_method="merged_16bit",
)
print(f"Merged model saved to: {FINAL_MODEL_DIR}")

---
### 7. (Optional) Push to Hugging Face Hub
Uncomment and set your HF repo to upload directly.

In [ ]:
# # Push LoRA adapter
# model.push_to_hub_merged(
#     "YOUR_USERNAME/Qwen2.5-7B-CyberSec-CPT-LoRA",
#     tokenizer,
#     save_method="lora",
#     token=os.getenv("HF_TOKEN"),
# )

# # Push merged model
# model.push_to_hub_merged(
#     "YOUR_USERNAME/Qwen2.5-7B-CyberSec-CPT",
#     tokenizer,
#     save_method="merged_16bit",
#     token=os.getenv("HF_TOKEN"),
# )

---
### Done!

Your model and checkpoints are saved to Google Drive at:
- **Checkpoints:** `MyDrive/LLM_Training/checkpoints/`
- **LoRA adapter:** `MyDrive/LLM_Training/lora_model/`
- **Merged model:** `MyDrive/LLM_Training/final_model/`

If Colab disconnects mid-training, just re-run the notebook — it will automatically resume from the latest checkpoint.